# Beyond-accuracy analysis: COSETTE/MARIUS vs SASRec++

Item-side distribution metrics (catalog coverage, Gini, normalized Shannon entropy, average recommendation popularity, APLT, novelty, intra-list diversity, MARIUS hallucination rate, popularity-decile exposure, and tail recall) that the base paper (arXiv:2508.14910) only shows informally (Fig 4 decile plot, Fig 8 collisions) but never tabulates.

Metrics are defined in `scripts/extensions/beyond_accuracy.py`. They run on the ranked Top-K lists that `sasrec.search` / `marius.search` already produce, so nothing in the authors' model or eval code changes.

**Data note.** Real numbers need the Top-K dumps produced on Snellius by `jobs/22_dump_topk_{beauty,sports}.sbatch` (a single read-only eval pass per model/seed). Part A below runs with no data so you can verify the metric implementations locally; Part B activates automatically once the dumps are under `reports/extensions/topk/<category>/`.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path.cwd()
while not (REPO / 'scripts' / 'extensions' / 'beyond_accuracy.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from scripts.extensions import beyond_accuracy as ba

DUMP_ROOT = REPO / 'reports' / 'extensions' / 'topk'
SEEDS = [42, 43, 44, 45, 46]
CATEGORIES = {'Beauty': 'beauty', 'Sports_and_Outdoors': 'sports'}
print('repo:', REPO)

## Part A. Synthetic sanity demo (no data required)
Three toy recommenders over a Zipfian catalog: a popularity-chaser, a uniform one, and a generative one with injected hallucinations. The metrics should order them as expected (chaser = high Gini, low coverage, low novelty, low APLT).

In [ ]:
rng = np.random.default_rng(0)
n_catalog, n_users, k = 1000, 2000, 10
item_pop = ((1.0 / np.arange(1, n_catalog + 1)) * 1e5).astype(np.int64) + 1
p = item_pop / item_pop.sum()
item_emb = rng.normal(size=(n_catalog, 32))
head50 = np.argsort(-item_pop)[:50]

recs = {
    'popularity-chaser': [list(rng.choice(head50, k, replace=False)) for _ in range(n_users)],
    'uniform':           [list(rng.choice(n_catalog, k, replace=False)) for _ in range(n_users)],
    'generative':        [[(-1 if rng.random() < 0.08 else int(x))
                            for x in rng.choice(n_catalog, k, replace=False, p=p)] for _ in range(n_users)],
}
demo = {name: ba.compute_all(r, item_pop, n_catalog, item_emb=item_emb, k=k) for name, r in recs.items()}
pd.DataFrame(demo).T[['coverage','gini','entropy_norm','arp','aplt','novelty','ild','hallucination_rate']].round(4)

## Part B. Real data (activates after the Snellius dump)
Loads the dumped Top-K and support tables, converts to catalog item indices (SASRec: vocab id minus special-token offset; MARIUS: semantic-ID tuple lookup, with unmatched tuples marked as hallucinations), and computes the suite per model and seed.

In [ ]:
def load_support(category):
    d = DUMP_ROOT / category
    meta = json.loads((d / 'meta.json').read_text())
    pop = np.load(d / 'popularity.npy')
    emb = np.load(d / 'embeddings.npy')
    t2i = json.loads((d / 'tuple_to_item.json').read_text())
    return meta, pop, emb, t2i

def load_model_recs(category, model, seed, meta, t2i):
    npz = np.load(DUMP_ROOT / category / f'{model}_seed{seed}_topk.npz')
    ns = meta['n_special']
    if model == 'sasrec':
        recs = [[int(v) - ns if int(v) >= ns else ba.HALLUCINATION for v in row] for row in npz['topk_items']]
        targets = [int(t) - ns if int(t) >= ns else ba.HALLUCINATION for t in npz['target_item']]
    else:
        recs = [[t2i.get(','.join(str(int(c)) for c in code), ba.HALLUCINATION) for code in row]
                for row in npz['topk_codes']]
        targets = [t2i.get(','.join(str(int(c)) for c in code), ba.HALLUCINATION) for code in npz['target_codes']]
    return recs, targets, npz['hist_len']

available = {c: (DUMP_ROOT / c / 'meta.json').exists() for c in CATEGORIES}
print('dumps present:', available)
if not any(available.values()):
    print('\nNo dumps yet. On Snellius run, per seed:')
    print('  sbatch --export=ALL,SEED=42 jobs/22_dump_topk_beauty.sbatch')
    print('  sbatch --export=ALL,SEED=42 jobs/22_dump_topk_sports.sbatch')
    print('then copy reports/extensions/topk/ back here.')

In [ ]:
def beyond_accuracy_table(category, ks=(10, 20)):
    meta, pop, emb, t2i = load_support(category)
    rows = []
    for model in ('sasrec', 'marius'):
        for seed in SEEDS:
            if not (DUMP_ROOT / category / f'{model}_seed{seed}_topk.npz').exists():
                continue
            recs, targets, _ = load_model_recs(category, model, seed, meta, t2i)
            for k in ks:
                m = ba.compute_all(recs, pop, meta['n_catalog'], item_emb=emb, targets=targets, k=k)
                m.update({'category': category, 'model': model, 'seed': seed})
                rows.append(m)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    metric_cols = [c for c in df.columns if c not in ('category','model','seed','k')]
    return df.groupby(['category','model','k'])[metric_cols].mean().round(4)

for c, present in available.items():
    if present:
        display(beyond_accuracy_table(c))

In [ ]:
# Popularity-decile exposure profile (absolute version of the paper's Fig 4).
def decile_table(category, k=10):
    meta, pop, emb, t2i = load_support(category)
    out = {}
    for model in ('sasrec', 'marius'):
        seeds = [s for s in SEEDS if (DUMP_ROOT / category / f'{model}_seed{s}_topk.npz').exists()]
        if not seeds:
            continue
        profiles = []
        for s in seeds:
            recs, _, _ = load_model_recs(category, model, s, meta, t2i)
            profiles.append(ba.popularity_decile_exposure(recs, pop, k=k))
        out[model] = np.mean(profiles, axis=0)
    if not out:
        return pd.DataFrame()
    return pd.DataFrame(out, index=[f'D{i+1}' for i in range(len(next(iter(out.values()))))]).round(4)

for c, present in available.items():
    if present:
        print(c, '- exposure share per popularity decile (D1=rarest, D10=most popular)')
        display(decile_table(c))

## What to read off Part B
- **Coverage / Gini / entropy / novelty / APLT / ILD:** does the generative model (MARIUS) spread exposure wider and surface more tail items than discriminative SASRec++, or the reverse? The paper hints (Fig 4) that MARIUS shifts toward mid-popular items; this quantifies it.
- **Tail recall:** does any diversity gain come with correct tail predictions, or just noise? (accuracy-meets-beyond-accuracy)
- **Hallucination rate:** MARIUS only; the share of generated tuples mapping to no item.
- **Decile profile:** the absolute exposure distribution; compare against the paper's Fig 4 difference plot.

Open scientific hook (needs a content-only RQ-VAE tokenizer variant): is COSETTE's *collaborative* tokenization itself a diversity/popularity-bias lever, vs content-only semantic IDs? Differentiate from Ghost (arXiv:2605.16825) and CRAB (arXiv:2604.05113), which target TIGER/LLM generative recommenders, not MARIUS.